# Okuno et al. (2025): connectome-only claims from the paper's deposits

A companion to `reproduce_okuno_2025.ipynb`. It tests a claim on the FlyWire connectome alone
(data elsewhere), from the paper's Zenodo record
[10.5281/zenodo.15532272](https://doi.org/10.5281/zenodo.15532272) (version of 2025-05-28), which
holds the PPSSI the paper computed for each FlyWire neuron (`data/flywire783_neuron_sepidx0sr140_3000mi1.mat`,
stored as int16 times 10,000, negative where no value; plotNeuralFC.m reads it so). The PPSSI is
not recomputed from the synapse table here. No random numbers are drawn.

| Claim | Data | Status |
|---|---|---|
| fig4b-ppssi-distribution | elsewhere | tested on the paper's per-neuron PPSSI |

In [ ]:
import io, warnings
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
import flyfc
warnings.filterwarnings("ignore")
m = sio.loadmat(io.BytesIO(flyfc.zenodo_member("data/flywire783_neuron_sepidx0sr140_3000mi1.mat")), variable_names=["Nspidx"])
nr = sio.loadmat(io.BytesIO(flyfc.zenodo_member("data/flywire783_neuron.mat")))
raw = m["Nspidx"].ravel()
print("neurons in the PPSSI table", raw.size, "| in the neuron table", nr["Nid"].size, "| stored range", raw.min(), "to", raw.max(), raw.dtype)

### fig4b-ppssi-distribution

> Over the 139,255 FlyWire neurons the PPSSI histogram resembles a normal distribution with a mean
> near 0.5, so many neurons have unsegregated synapses.

Reproduced when the neuron count matches (139,255, tolerance 0) and the mean PPSSI is within 0.4
to 0.6.

In [ ]:
x = raw.astype(float) / 10000
valid = x >= 0
print("neurons with a PPSSI", int(valid.sum()), "| without (stored negative)", int((~valid).sum()), "| expected 139,255")
v = x[valid]
print(f"PPSSI mean {v.mean():.4f}, median {np.median(v):.4f}, SD {v.std():.4f}, skewness {((v - v.mean())**3).mean() / v.std()**3:.3f}")
readme = flyfc.code_file("README.md").decode()
print("code README line:", [l for l in readme.splitlines() if "full separation" in l])
print("neurons with PPSSI exactly 1 (fully segregated)", int((v == 1).sum()), "| exactly 0", int((v == 0).sum()))
h, e = np.histogram(v, bins=np.arange(0, 1.0001, 0.05))
for a, b in zip(e[:-1], h):
    print(f"{a:.2f}-{a + 0.05:.2f} {b}")
plt.figure(figsize=(6, 3)); plt.bar(e[:-1] + 0.025, h, width=0.05); plt.xlabel("PPSSI"); plt.ylabel("neurons"); plt.show()
print("all", int(valid.sum()), "values in [0, 1] are shown")

**Result: partially reproduced.** The table holds 139,255 neurons, the paper's count, but 1,111 of
them carry no PPSSI, so 138,144 neurons have one (the README of the paper's code prints 6154/138144 for
the fully segregated neurons, the same denominator). The mean PPSSI is 0.471 (median 0.451), within 0.4 to 0.6. The mean holds; the count of
neurons with a PPSSI is 0.8% below the stated number, which the tolerance of 0 does not allow.